# From Crystal Structures to ML-Ready Layer Descriptors
## A portfolio demonstration of the `squarenet_ml` data-collection pipeline

**Repository:** `cookms/squarenet_ml`  
**Notebook role:** Stage 1 of a materials-informatics project that tests whether geometric and chemical layer information improves prediction of stability, formation energy, metallicity, band gap, and topological-semimetal labels.

This notebook is deliberately structured as a small, reproducible data product rather than a one-off script. It demonstrates the scientific logic, software interface, output schema, quality checks, and the connection to downstream machine learning.

## What this notebook demonstrates

1. **Offline detector validation:** create ideal and distorted layered crystal structures and verify that the square-net detector responds to controlled geometric changes.
2. **Layer-level feature extraction:** convert detector results into tidy tabular records suitable for exploratory analysis and future multi-instance learning.
3. **Optional Materials Project collection:** run a small, explicit-ID query through the repository's production pipeline without exposing an API key or accidentally launching a large screen.
4. **Data quality and ML readiness:** inspect keys, missingness, class balance, and available targets before modeling.
5. **Research roadmap:** define rigorous baseline, feature-ablation, topological-classification, and layer-level multi-instance-learning experiments.

> The notebook runs its deterministic synthetic example without network access. The Materials Project section is opt-in and remains disabled until `RUN_MP_DEMO = True` and an API key is available in the environment.

## Project architecture

The repository supports two related levels of analysis:

- **Low-level detector:** `find_square_net_planes(structure, ...)` returns one candidate record per `(axis, plane_id, species)` layer. These records contain geometric scores, adjacent-plane context, pass/fail labels, and optional CrystalNN chemistry/bonding features.
- **Materials Project pipeline:** `run_pipeline(config)` queries structures and properties, preprocesses each crystal, runs the detector, and writes compact material-level and axis/species-level tables.

The low-level records are the natural **instances** for a future multi-instance-learning model. The material-level record is the **bag label** and property target.

## Environment setup

Run this notebook from a clone of the repository after installing its dependencies:

```bash
git clone https://github.com/cookms/squarenet_ml.git
cd squarenet_ml
python -m venv .venv
# Activate the environment, then:
pip install -r requirements.txt
jupyter lab
```

For the optional Materials Project section, store the key in an environment variable rather than in the notebook:

```bash
export MP_API_KEY="your-key"
```

On PowerShell:

```powershell
$env:MP_API_KEY = "your-key"
```

In [1]:
from __future__ import annotations

import importlib.util
import json
import os
import sys
from dataclasses import asdict, is_dataclass
from pathlib import Path
from typing import Any, Iterable


def find_repo_root(start: Path | None = None) -> Path:
    """Locate the repository root, with an environment override for portability."""
    override = os.environ.get("SQUARENET_ML_ROOT")
    start = Path(override).expanduser() if override else (start or Path.cwd())
    start = start.resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "squarenet").is_dir() and (candidate / "requirements.txt").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the squarenet_ml repository root. Place this notebook inside "
        "the cloned repository or set SQUARENET_ML_ROOT to the clone directory."
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
    "pymatgen": "pymatgen",
}
missing = [pip_name for module, pip_name in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ModuleNotFoundError(
        "Missing required packages: " + ", ".join(missing) +
        f". Install the repository requirements with: pip install -r {REPO_ROOT / 'requirements.txt'}"
    )

print(f"Repository root: {REPO_ROOT}")

Repository root: C:\Users\mscoo\GIT_projects\Squarenet_ML


In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from pymatgen.core import Lattice, Structure

from squarenet.detect import find_square_net_planes
from squarenet.visualization import plot_detection_summary

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

OUTPUT_ROOT = REPO_ROOT / "outputs" / "data_collection_demo"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"Notebook outputs will be written to: {OUTPUT_ROOT}")

Notebook outputs will be written to: C:\Users\mscoo\GIT_projects\Squarenet_ML\outputs\data_collection_demo


# 1. Deterministic detector validation

A portfolio notebook should not depend entirely on an external API. We first create two controlled structures:

- **Ideal square net:** equal in-plane lattice constants and 90-degree angles.
- **Rectangular distortion:** one in-plane lattice constant is stretched, breaking the equal-neighbor-length condition while preserving a layered structure.

Both structures contain a Si layer separated from a Ca spacer layer. The detector is restricted to the Si species and the crystallographic `c` axis so that the comparison isolates the intended geometric perturbation.

In [56]:
def make_layered_test_structure(
    a: float,
    b: float,
    c: float,
    alpha: float = 90.0,
    beta: float = 90.0,
    gamma: float = 90.0,
    in_plane_repeat: tuple[int, int, int] = (2, 2, 1),
) -> Structure:
    """Build a layered Si/Ca test structure with a tunable Si-plane metric.

    The Si atoms form a two-dimensional Bravais net in the a-b plane.

    Parameters
    ----------
    a, b, c
        Lattice-vector lengths in angstroms.
    alpha
        Angle between b and c, in degrees.
    beta
        Angle between a and c, in degrees.
    gamma
        Angle between a and b, in degrees. This controls the in-plane
        angle of the Si net.
    in_plane_repeat
        Supercell transformation applied after constructing the unit cell.
    """
    lattice = Lattice.from_parameters(
        a=a,
        b=b,
        c=c,
        alpha=alpha,
        beta=beta,
        gamma=gamma,
    )

    structure = Structure(
        lattice=lattice,
        species=["Si", "Ca"],
        coords=[
            [0.0, 0.0, 0.5],  # candidate Si layer
            [0.5, 0.5, 0.0],  # chemically distinct spacer layer
        ],
        coords_are_cartesian=False,
    )

    structure.make_supercell(in_plane_repeat)
    return structure


structures = {
    # Exact square net:
    # a = b and gamma = 90 degrees.
    "ideal_square": make_layered_test_structure(
        a=3.35,
        b=3.35,
        c=9.0,
        gamma=90.0,
    ),

    # Equal neighbor distances, but slightly non-square angles.
    # This is a rhombic distortion.
    "nearly_square_angle": make_layered_test_structure(
        a=3.35,
        b=3.35,
        c=9.0,
        gamma=87.0,
    ),

    # Slightly unequal neighbor distances but exact right angles.
    # This is a weak rectangular distortion.
    "nearly_square_lengths": make_layered_test_structure(
        a=3.35,
        b=3.45,
        c=9.0,
        gamma=90.0,
    ),

    # Both the lengths and angles are slightly distorted.
    "nearly_square_combined": make_layered_test_structure(
        a=3.35,
        b=3.45,
        c=9.0,
        gamma=87.0,
    ),

    # Clear rectangular failure:
    # right angles, but strongly unequal neighbor distances.
    "rectangular_failure": make_layered_test_structure(
        a=3.35,
        b=5.00,
        c=9.0,
        gamma=90.0,
    ),

    # Equal bond lengths but a clearly non-square in-plane angle.
    "angular_failure": make_layered_test_structure(
        a=3.35,
        b=3.35,
        c=9.0,
        gamma=72.0,
    ),

    # Both criteria should clearly fail.
    "combined_failure": make_layered_test_structure(
        a=3.35,
        b=4.50,
        c=9.0,
        gamma=72.0,
    ),
}

structure_overview = pd.DataFrame(
    [
        {
            "example": name,
            "formula": structure.composition.reduced_formula,
            "n_sites": len(structure),
            "a_A": structure.lattice.a,
            "b_A": structure.lattice.b,
            "c_A": structure.lattice.c,
            "b_over_a": structure.lattice.b / structure.lattice.a,
        }
        for name, structure in structures.items()
    ]
)

structure_overview

,example,formula,n_sites,a_A,b_A,c_A,b_over_a
0,ideal_square,CaSi,8,6.7,6.7,9.0,1.000000
1,nearly_square_angle,CaSi,8,6.7,6.7,9.0,1.000000
2,nearly_square_lengths,CaSi,8,6.7,6.9,9.0,1.029851
3,nearly_square_combined,CaSi,8,6.7,6.9,9.0,1.029851
4,rectangular_failure,CaSi,8,6.7,10.0,9.0,1.492537
5,angular_failure,CaSi,8,6.7,6.7,9.0,1.000000
6,combined_failure,CaSi,8,6.7,9.0,9.0,1.343284


In [57]:
DETECTOR_KWARGS = dict(
    axes=("c",),
    species=("Si",),
    plane_tol=0.02,
    k_nn=9,
    len_tol=0.10,
    ang_tol_deg=7.0,
    min_pass_fraction=0.50,
    score_threshold=0.50,
    return_all=True,
    preserve_visualization_data=True,
    adjacent_by="atom",
    nn_intra_min_max=5.5,
    min_adj_dist_any_atom_min=2.0,
    forbid_coplane_mixed_species=True,
    isolate_same_species_adjacent=True,
    isolate_same_species_adjacent_dist_min=2.0,
    # Keep the synthetic geometry test fast and independent of oxidation-state guessing.
    enforce_no_out_of_plane_same_species_bonds=False,
    compute_crystalnn_features=False,
)


def result_to_record(result: Any, *, include_visualization: bool = False) -> dict[str, Any]:
    """Convert a detector result to a dictionary without recursively copying plot arrays."""
    if hasattr(result, "__dict__"):
        record = dict(vars(result))
    elif is_dataclass(result):
        record = asdict(result)
    else:
        record = dict(result)

    if not include_visualization:
        record.pop("visualization_data", None)
    return record


def results_to_frame(results: Iterable[Any]) -> pd.DataFrame:
    return pd.DataFrame([result_to_record(result) for result in results])


raw_results = {
    name: find_square_net_planes(structure, **DETECTOR_KWARGS)
    for name, structure in structures.items()
}

for name, results in raw_results.items():
    print(f"{name}: {len(results)} candidate layer record(s)")

ideal_square: 1 candidate layer record(s)
nearly_square_angle: 1 candidate layer record(s)
nearly_square_lengths: 1 candidate layer record(s)
nearly_square_combined: 1 candidate layer record(s)
rectangular_failure: 1 candidate layer record(s)
angular_failure: 1 candidate layer record(s)
combined_failure: 1 candidate layer record(s)


In [47]:
layer_frames = []
for example, results in raw_results.items():
    frame = results_to_frame(results)
    frame.insert(0, "example", example)
    layer_frames.append(frame)

synthetic_layers = pd.concat(layer_frames, ignore_index=True)

preferred_columns = [
    "example",
    "axis",
    "plane_id",
    "species",
    "n_sites",
    "passes",
    "passes2",
    "pass_fraction",
    "mean_score",
    "median_score",
    "nn_intra_min",
    "nn_intra_mean",
    "tol_ratio_any",
    "uv_len_err_mean",
    "uv_ang_deg_mean",
    "min_adj_dist_any_atom",
    "passes2_fail_reasons",
]
visible_columns = [column for column in preferred_columns if column in synthetic_layers.columns]
synthetic_layers[visible_columns].sort_values(["example", "mean_score"], ascending=[True, False])

,example,axis,plane_id,species,n_sites,passes,passes2,pass_fraction,mean_score,median_score,nn_intra_min,nn_intra_mean,tol_ratio_any,uv_len_err_mean,uv_ang_deg_mean,min_adj_dist_any_atom,passes2_fail_reasons
5,angular_failure,c,1,Si,4,False,False,0.0,1.343812e-03,1.343812e-03,3.35,3.35,0.682010,1.325639e-16,99.0,4.911953,[primary_pass_failed]
6,combined_failure,c,1,Si,4,False,False,0.0,2.512615e-07,2.512615e-07,3.35,3.35,0.659672,2.929936e-01,108.0,5.078278,[primary_pass_failed]
0,ideal_square,c,1,Si,4,True,True,1.0,1.000000e+00,1.000000e+00,3.35,3.35,0.658749,0.000000e+00,90.0,5.085396,[]
1,nearly_square_angle,c,1,Si,4,True,True,1.0,8.322075e-01,8.322075e-01,3.35,3.35,0.662522,1.325639e-16,90.0,5.056439,[]
3,nearly_square_combined,c,1,Si,4,True,True,1.0,7.632431e-01,7.632431e-01,3.35,3.35,0.660442,2.941176e-02,90.0,5.072358,[]
2,nearly_square_lengths,c,1,Si,4,True,True,1.0,9.171308e-01,9.171308e-01,3.35,3.35,0.656595,2.941176e-02,90.0,5.102083,[]
4,rectangular_failure,c,1,Si,4,False,False,0.0,1.647130e-07,1.647130e-07,3.35,3.35,0.618827,3.952096e-01,90.0,5.413467,[primary_pass_failed]


### Expected interpretation

The ideal structure should rank above the rectangular distortion because the detector rewards four approximately equivalent in-plane neighbors arranged along two orthogonal directions. The comparison is more informative than a single positive example: it tests whether the score changes in the physically expected direction under a controlled perturbation.

`passes` reflects the primary geometric screen. `passes2` starts from that result and applies additional distance, composition, and optional bonding constraints.

In [48]:
def count_truthy(group: pd.DataFrame, column: str) -> int:
    if column not in group.columns:
        return 0
    return int(group[column].fillna(False).astype(bool).sum())


def summarize_example(group: pd.DataFrame) -> pd.Series:
    ordered = group.sort_values(["passes2", "mean_score"], ascending=[False, False])
    best = ordered.iloc[0]
    return pd.Series(
        {
            "n_candidates": len(group),
            "n_passes": count_truthy(group, "passes"),
            "n_passes2": count_truthy(group, "passes2"),
            "best_mean_score": best.get("mean_score", np.nan),
            "best_pass_fraction": best.get("pass_fraction", np.nan),
            "best_nn_intra_min_A": best.get("nn_intra_min", np.nan),
            "best_tol_ratio_any": best.get("tol_ratio_any", np.nan),
        }
    )


summary_records = {
    example: summarize_example(group)
    for example, group in synthetic_layers.groupby("example", dropna=False)
}
synthetic_summary = pd.DataFrame.from_dict(summary_records, orient="index")
synthetic_summary.index.name = "example"
synthetic_summary

,n_candidates,n_passes,n_passes2,best_mean_score,best_pass_fraction,best_nn_intra_min_A,best_tol_ratio_any
example,,,,,,,
angular_failure,1.0,0.0,0.0,1.343812e-03,0.0,3.35,0.682010
combined_failure,1.0,0.0,0.0,2.512615e-07,0.0,3.35,0.659672
ideal_square,1.0,1.0,1.0,1.000000e+00,1.0,3.35,0.658749
nearly_square_angle,1.0,1.0,1.0,8.322075e-01,1.0,3.35,0.662522
nearly_square_combined,1.0,1.0,1.0,7.632431e-01,1.0,3.35,0.660442
nearly_square_lengths,1.0,1.0,1.0,9.171308e-01,1.0,3.35,0.656595
rectangular_failure,1.0,0.0,0.0,1.647130e-07,0.0,3.35,0.618827


In [1]:
ax = synthetic_summary[["best_mean_score", "best_pass_fraction"]].plot(
    kind="bar",
    figsize=(8, 4),
    rot=0,
    title="Controlled detector response: ideal vs. distorted layer",
)
ax.set_xlabel("")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

SyntaxError: '[' was never closed (1261617793.py, line 1)

## Detector explanation plot

The visualization module can show the projected candidate layer, selected in-plane neighbors, local scores, and nearby planes. This is valuable for both scientific debugging and model interpretability: a candidate should not be accepted solely because a scalar score is high.

In [2]:
def best_result(results: list[Any]) -> Any:
    def rank_key(result: Any) -> tuple[int, float]:
        score = float(getattr(result, "mean_score", float("-inf")))
        if not np.isfinite(score):
            score = float("-inf")
        return int(bool(getattr(result, "passes2", False))), score

    return max(results, key=rank_key)


if raw_results["ideal_square"]:
    ideal_best = best_result(raw_results["ideal_square"])
    fig, _axes = plot_detection_summary(
        structures["ideal_square"],
        ideal_best,
        representative_site="worst",
    )
    fig.suptitle("Ideal synthetic square-net candidate", y=1.02)
    plt.show()
else:
    print("No ideal-square candidates were returned; inspect detector settings before plotting.")

NameError: name 'Any' is not defined

In [51]:
def serialize_nested(value: Any) -> Any:
    if isinstance(value, np.ndarray):
        return json.dumps(value.tolist(), default=str)
    if isinstance(value, (dict, list, tuple)):
        return json.dumps(value, default=str)
    return value


synthetic_export = synthetic_layers.copy()
for column in synthetic_export.columns:
    if synthetic_export[column].map(
        lambda value: isinstance(value, (dict, list, tuple, np.ndarray))
    ).any():
        synthetic_export[column] = synthetic_export[column].map(serialize_nested)

synthetic_path = OUTPUT_ROOT / "synthetic_layer_candidates.csv"
synthetic_export.to_csv(synthetic_path, index=False)
print(f"Wrote {len(synthetic_export)} layer records to {synthetic_path}")

Wrote 7 layer records to C:\Users\mscoo\GIT_projects\Squarenet_ML\outputs\data_collection_demo\synthetic_layer_candidates.csv


# 2. What information is collected?

The detector creates a rich layer representation rather than a single binary label.

| Feature family | Representative fields | Scientific role |
|---|---|---|
| Square geometry | `mean_score`, `pass_fraction`, `uv_len_err_mean`, `uv_ang_deg_mean` | Quantifies how closely a candidate resembles a 2D square lattice |
| Intralayer scale | `nn_intra_min`, `nn_intra_mean`, `tol_ratio_any` | Characterizes net size and geometric tolerance |
| Layer isolation | `min_adj_dist_any_atom`, `min_adj_dist_any_plane`, plane separation metrics | Distinguishes isolated nets from 3D-connected motifs |
| Co-plane chemistry | species counts, major-species fraction, mixed-plane flags | Captures chemical purity and substitution within the candidate layer |
| Adjacent-layer chemistry | nearest-plane species counts and identities | Encodes the local chemical environment surrounding the net |
| CrystalNN bonding | coordination, in-plane/out-of-plane neighbor species, bond lengths and angles | Adds chemically informed connectivity beyond pure geometry |
| Screening labels | `passes`, `passes2`, failure reasons | Supports supervised detector emulation, auditing, and threshold studies |

This representation enables explicit feature-ablation experiments: composition-only, global-structure, layer-geometry, layer-chemistry, and combined models.

# 3. Optional Materials Project data collection

The next section demonstrates the repository's production workflow:

1. query selected Materials Project summaries,
2. fetch structures,
3. standardize the structure used for detection,
4. detect and summarize candidate layers,
5. write `materials.csv`, `axis_species.csv`, `meta.json`, and a processed-ID log.

The demo uses the repository's `mpids.txt` when available. Those IDs are suitable as smoke tests, not as a scientifically balanced benchmark. A final study should construct a documented cohort with known positives, hard negatives, composition controls, and prototype-aware splits.

In [ ]:
from squarenet.config import (
    DetectConfig,
    MPQueryConfig,
    OutputConfig,
    PipelineConfig,
    PreprocessConfig,
)
from squarenet.pipeline import run_pipeline


def read_material_ids(path: Path) -> list[str]:
    if not path.exists():
        return []
    ids = []
    for line in path.read_text(encoding="utf-8").splitlines():
        stripped = line.strip()
        if stripped and not stripped.startswith("#"):
            ids.append(stripped)
    return ids


RUN_MP_DEMO = False  # Deliberate safety switch: set True only when ready.
MP_API_KEY = os.environ.get("MP_API_KEY") or os.environ.get("MAPI_KEY")
DEMO_MATERIAL_IDS = read_material_ids(REPO_ROOT / "mpids.txt") or ["mp-149", "mp-13"]
MP_OUTPUT_DIR = OUTPUT_ROOT / "materials_project_demo"

print(f"Materials Project key detected: {bool(MP_API_KEY)}")
print(f"Demo IDs: {DEMO_MATERIAL_IDS}")

In [ ]:
mp_demo_config = PipelineConfig(
    mp=MPQueryConfig(
        api_key=MP_API_KEY,
        material_ids=DEMO_MATERIAL_IDS,
        limit=len(DEMO_MATERIAL_IDS),
    ),
    preprocess=PreprocessConfig(
        # Conventional cells make this small notebook demo fast and transparent.
        # A production sweep can switch to "processed" and configure a supercell.
        structure_source="conventional",
        to_conventional=True,
        symprec=1e-3,
        angle_tolerance=5.0,
        supercell=None,
        sym_supercell=None,
    ),
    detect=DetectConfig(
        species=None,
        axes=("c", "a", "b"),
        plane_tol=0.01,
        k_nn=9,
        len_tol=0.10,
        ang_tol_deg=5.0,
        min_pass_fraction=0.55,
        score_threshold=0.50,
        return_all=True,
        preserve_visualization_data=False,
        adjacent_by="atom",
        nn_intra_min_max=4.0,
        min_adj_dist_any_atom_min=2.0,
        forbid_coplane_mixed_species=True,
        isolate_same_species_adjacent=True,
        isolate_same_species_adjacent_dist_min=2.0,
        enforce_no_out_of_plane_same_species_bonds=True,
        compute_crystalnn_features=True,
        crystalnn_weight_cutoff=0.0,
    ),
    output=OutputConfig(
        out_dir=str(MP_OUTPUT_DIR),
        write_csv=True,
        # CSV is sufficient for a two-material demo and avoids requiring optional pyarrow.
        write_parquet=False,
        resume=True,
        skip_existing=True,
        export_cifs=False,
        flush_every=1,
    ),
    meta={
        "notebook": "01_square_net_data_collection_demo.ipynb",
        "purpose": "portfolio data-collection demonstration",
        "random_state": RANDOM_STATE,
    },
)

mp_demo_config

In [ ]:
if RUN_MP_DEMO:
    if not MP_API_KEY:
        raise RuntimeError(
            "RUN_MP_DEMO is True, but no API key was found. "
            "Set MP_API_KEY in the environment and restart the kernel."
        )
    materials_df, axis_species_df = run_pipeline(mp_demo_config)
else:
    print("Materials Project collection skipped. Set RUN_MP_DEMO = True to execute it.")
    materials_path = MP_OUTPUT_DIR / "materials.csv"
    axis_species_path = MP_OUTPUT_DIR / "axis_species.csv"
    materials_df = pd.read_csv(materials_path) if materials_path.exists() else pd.DataFrame()
    axis_species_df = pd.read_csv(axis_species_path) if axis_species_path.exists() else pd.DataFrame()

print(f"materials_df shape: {materials_df.shape}")
print(f"axis_species_df shape: {axis_species_df.shape}")

## Output granularity

- `materials_df`: one row per material, centered on the detector's dominant candidate layer and augmented with selected Materials Project properties.
- `axis_species_df`: one row per `(material_id, axis, species)`, centered on the best layer for that axis/species combination.
- Low-level detector results: one row per candidate layer. The synthetic section above shows this schema directly.

**Important future-MIL engineering task:** persist every low-level candidate layer from the Materials Project pipeline to a dedicated `layers.parquet` table. The current pipeline's public return value contains the material and axis/species summaries, while multi-instance learning requires the full candidate-layer bag for every material.

In [ ]:
def compact_preview(df: pd.DataFrame, preferred: list[str], n: int = 8) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame({"status": ["No rows available; run the optional collection section."]})
    columns = [column for column in preferred if column in df.columns]
    return df[columns].head(n) if columns else df.head(n)


display(Markdown("### Material-level table"))
display(
    compact_preview(
        materials_df,
        [
            "material_id",
            "formula",
            "sg_symbol",
            "crystal_system",
            "has_any_pass",
            "dominant_axis",
            "dominant_species",
            "dominant_mean_score",
            "dominant_nn_intra_min",
            "energy_above_hull",
            "band_gap",
        ],
    )
)

display(Markdown("### Axis/species-level table"))
display(
    compact_preview(
        axis_species_df,
        [
            "material_id",
            "formula",
            "axis",
            "species",
            "n_layers",
            "n_pass",
            "best_layer_has_pass",
            "best_layer_mean_score",
            "best_layer_nn_intra_min",
            "energy_above_hull",
            "band_gap",
        ],
    )
)

# 4. Data-quality audit

Before fitting any model, verify that the table has a stable primary key, sensible dtypes, acceptable missingness, and a target distribution compatible with the planned evaluation metric. This prevents a polished model from masking a flawed dataset.

In [ ]:
def audit_table(df: pd.DataFrame, name: str, key_columns: list[str]) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(
            [{
                "table": name,
                "rows": 0,
                "columns": 0,
                "duplicate_keys": np.nan,
                "missing_cells_fraction": np.nan,
                "all_null_columns": 0,
            }]
        )

    missing_keys = [column for column in key_columns if column not in df.columns]
    duplicate_keys = (
        np.nan if missing_keys else int(df.duplicated(subset=key_columns, keep=False).sum())
    )
    missing_fraction = float(df.isna().sum().sum() / max(df.size, 1))
    all_null_columns = int(df.isna().all(axis=0).sum())

    return pd.DataFrame(
        [{
            "table": name,
            "rows": len(df),
            "columns": df.shape[1],
            "duplicate_keys": duplicate_keys,
            "missing_cells_fraction": missing_fraction,
            "all_null_columns": all_null_columns,
        }]
    )


audits = pd.concat(
    [
        audit_table(synthetic_layers, "synthetic_layers", ["example", "axis", "plane_id", "species"]),
        audit_table(materials_df, "materials", ["material_id"]),
        audit_table(axis_species_df, "axis_species", ["material_id", "axis", "species"]),
    ],
    ignore_index=True,
)
audits

In [ ]:
def missingness_report(df: pd.DataFrame, top_n: int = 15) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=["column", "missing_count", "missing_fraction", "dtype"])
    report = pd.DataFrame(
        {
            "column": df.columns,
            "missing_count": df.isna().sum().values,
            "missing_fraction": df.isna().mean().values,
            "dtype": df.dtypes.astype(str).values,
        }
    )
    return report.sort_values(["missing_fraction", "missing_count"], ascending=False).head(top_n)


missingness_report(materials_df)

In [ ]:
if not materials_df.empty and "has_any_pass" in materials_df.columns:
    balance_values = materials_df["has_any_pass"].astype("object")
    balance_values = balance_values.where(balance_values.notna(), "missing")
    class_balance = (
        balance_values
        .value_counts(dropna=False)
        .rename_axis("has_any_pass")
        .to_frame("count")
    )
    class_balance["fraction"] = class_balance["count"] / class_balance["count"].sum()
    display(class_balance)
else:
    print("Class-balance summary will appear after Materials Project data are collected.")

# 5. Build an initial ML-ready material table

The current pipeline directly includes `band_gap` and `energy_above_hull`. Two useful demonstration targets can be derived transparently:

- **Metallicity proxy:** `band_gap <= 1e-6 eV`.
- **Near-hull stability label:** `energy_above_hull <= 0.05 eV/atom`.

These are modeling conveniences, not universal physical definitions. The threshold must be stated, sensitivity-tested, and kept separate from the detector labels.

Formation energy is not currently propagated into the pipeline output. Adding `formation_energy_per_atom` to the Materials Project field list and output propagation is a concrete next repository enhancement.

In [ ]:
def build_ml_material_table(materials: pd.DataFrame) -> pd.DataFrame:
    if materials.empty:
        return materials.copy()

    table = materials.copy()

    if "band_gap" in table.columns:
        band_gap = pd.to_numeric(table["band_gap"], errors="coerce")
        metal_proxy = pd.Series(pd.NA, index=table.index, dtype="Int64")
        known_band_gap = band_gap.notna()
        metal_proxy.loc[known_band_gap] = (band_gap.loc[known_band_gap] <= 1e-6).astype(int)
        table["target_is_metal_proxy"] = metal_proxy

    if "energy_above_hull" in table.columns:
        e_hull = pd.to_numeric(table["energy_above_hull"], errors="coerce")
        near_hull = pd.Series(pd.NA, index=table.index, dtype="Int64")
        known_e_hull = e_hull.notna()
        near_hull.loc[known_e_hull] = (e_hull.loc[known_e_hull] <= 0.05).astype(int)
        table["target_near_hull_50meV"] = near_hull

    if "material_id" in table.columns:
        table = table.drop_duplicates(subset=["material_id"], keep="last")

    return table


ml_materials = build_ml_material_table(materials_df)

identifier_columns = [
    "material_id",
    "formula",
    "sg_number",
    "sg_symbol",
    "crystal_system",
]
property_columns = [
    "energy_above_hull",
    "band_gap",
    "target_is_metal_proxy",
    "target_near_hull_50meV",
]
layer_feature_columns = [
    column
    for column in ml_materials.columns
    if column.startswith("dominant_") or column in {"n_layers_total", "n_axes", "n_species", "n_pass", "pass_fraction_total"}
]
selected_columns = [
    column
    for column in identifier_columns + property_columns + layer_feature_columns
    if column in ml_materials.columns
]

ml_materials[selected_columns].head() if selected_columns else ml_materials.head()

In [ ]:
if not ml_materials.empty and {"dominant_mean_score", "band_gap"}.issubset(ml_materials.columns):
    plot_df = ml_materials[["dominant_mean_score", "band_gap", "has_any_pass"]].copy()
    plot_df["dominant_mean_score"] = pd.to_numeric(plot_df["dominant_mean_score"], errors="coerce")
    plot_df["band_gap"] = pd.to_numeric(plot_df["band_gap"], errors="coerce")
    plot_df = plot_df.dropna(subset=["dominant_mean_score", "band_gap"])

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for label, group in plot_df.groupby("has_any_pass", dropna=False):
        ax.scatter(group["dominant_mean_score"], group["band_gap"], label=f"has_any_pass={label}", alpha=0.8)
    ax.set_xlabel("Dominant layer mean square-net score")
    ax.set_ylabel("Materials Project band gap (eV)")
    ax.set_title("First-look relationship between layer geometry and band gap")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("The score-vs-property plot will appear after a Materials Project dataset is collected.")

In [ ]:
if not ml_materials.empty:
    ml_table_path = OUTPUT_ROOT / "ml_materials_demo.csv"
    schema_path = OUTPUT_ROOT / "ml_materials_demo_schema.json"

    ml_materials.to_csv(ml_table_path, index=False)
    schema = {
        "primary_key": ["material_id"],
        "n_rows": int(len(ml_materials)),
        "columns": {column: str(dtype) for column, dtype in ml_materials.dtypes.items()},
        "candidate_targets": [
            column
            for column in [
                "energy_above_hull",
                "band_gap",
                "target_is_metal_proxy",
                "target_near_hull_50meV",
            ]
            if column in ml_materials.columns
        ],
        "layer_feature_columns": layer_feature_columns,
        "notes": [
            "Metallicity and near-hull labels are notebook-derived proxies.",
            "Formation energy requires an additional Materials Project field in the repository pipeline.",
            "Full layer bags should be persisted before multi-instance learning.",
        ],
    }
    schema_path.write_text(json.dumps(schema, indent=2), encoding="utf-8")

    print(f"Wrote ML table: {ml_table_path}")
    print(f"Wrote schema:   {schema_path}")
else:
    print("No Materials Project rows are available, so only the synthetic layer table was exported.")

# 6. Machine-learning experiment roadmap

## Experiment A — Do layer descriptors improve property prediction?

Compare nested feature sets under identical splits and preprocessing:

1. **Baseline:** composition descriptors and inexpensive global crystal descriptors.
2. **Baseline + detector label:** add `has_any_pass` and dominant species/axis.
3. **Baseline + layer geometry:** add scores, in-plane distances, angle/length errors, and layer separations.
4. **Baseline + layer chemistry:** add co-plane, adjacent-plane, oxidation-state, and CrystalNN bonding summaries.
5. **Combined:** all feature families.

Targets:

- regression: band gap, energy above hull, formation energy per atom;
- classification: metal/nonmetal, near-hull stability, optional synthesized/experimental status.

Report repeated grouped cross-validation, uncertainty, and paired fold-level differences between feature sets rather than only a single best score.

## Experiment B — Can the collected descriptors reproduce topological-semimetal classifications?

Join a carefully versioned external label source to Materials Project IDs or structure-matched entries. Evaluate:

- a composition/global-structure baseline;
- square-net geometry only;
- geometry + chemical environment;
- full combined features.

Use PR-AUC and calibrated probabilities for imbalanced labels. Audit label-source coverage and structure-matching ambiguity before interpreting performance.

## Leakage-resistant evaluation design

Random row splits will overestimate performance when closely related compositions or structure prototypes appear in both train and test sets. Recommended safeguards:

- group by reduced formula, chemical system, or structure prototype;
- keep polymorphs and near-duplicate structures in the same fold;
- fit all imputers, encoders, scalers, and feature selectors inside the cross-validation pipeline;
- tune detector thresholds only on training data when the detector output itself is treated as a learned design choice;
- reserve a chemically distinct external test set for the final portfolio result;
- report bootstrap confidence intervals and feature-ablation deltas.

For topological labels, also prevent leakage from family names, hand-curated labels, or database fields that directly encode the target.

# 7. Future layer-level multi-instance learning

A material can contain many candidate layers, only some of which may control the property of interest. This naturally defines a multi-instance-learning problem:

- **Bag:** one material.
- **Instances:** all candidate `(axis, plane_id, species)` layers.
- **Instance features:** geometry, layer separation, chemistry, CrystalNN bonding, and failure diagnostics.
- **Bag context:** composition and global crystal descriptors.
- **Target:** material property or topological class.

A practical first architecture is an instance encoder followed by gated-attention pooling and a bag-level prediction head. Compare it against simple aggregations such as maximum score, mean pooling, top-k pooling, and the existing dominant-layer summary. Attention weights can identify which layers the model considered important, but they should be treated as diagnostic evidence rather than guaranteed causal explanations.

### Data-engineering requirement

Add a stable layer key such as:

```text
(material_id, structure_version, axis, plane_id, plane_center_frac, species)
```

and persist a partitioned `layers.parquet` table. Keep detector configuration and code version in metadata so that layer bags are reproducible.

# Conclusions

This notebook establishes a credible first stage for the larger project:

- the detector is tested against a controlled geometric perturbation;
- candidate layers are converted into inspectable, exportable records;
- the Materials Project pipeline is configured safely and reproducibly;
- output tables are audited before modeling;
- missing targets and the layer-persistence requirement are made explicit;
- the downstream experiments are framed as feature-ablation and generalization tests rather than unconstrained model fitting.

The strongest next repository enhancement is to persist the complete per-layer table during batch Materials Project runs, while adding formation energy and any additional target metadata to the material-level output.